In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 
import scipy.stats as stats
from scipy import signal

### Helpers 

In [ ]:
def plot_trajectories_barriers(trajectories: np.array, barrier: list, plt_title: str):
    probe_size = trajectories.shape[0]
    plt.figure(figsize=(12, 6))

    # Plot individual trajectories
    for i in range(probe_size):
        plt.plot(trajectories[i, :], alpha=0.7, linewidth=1.5, 
                 label=f'Sample {i+1}' if probe_size <= 5 else None)

    # Horizontal lines representing statistical boundaries (e.g., 5 * sigma_stationary)
    plt.axhline(y=barrier[0], color='red', linestyle='--', label='Lower Barrier')
    plt.axhline(y=barrier[1], color='red', linestyle='--', label='Upper Barrier')

    plt.title(plt_title, fontsize=14)
    plt.xlabel('Time (Step)', fontsize=12)
    plt.ylabel('Position (State)', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.ylim(barrier[0] - 1, barrier[1] + 1) 

    if probe_size <= 10:
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()

def plot_ar_comparison(probes, simulations_params, burn_in=50):
    """
    Plots the Kernel Density Estimation (KDE) to visualize the stationary distribution.
    """
    plt.figure(figsize=(12, 7))
    
    for data, params in zip(probes, simulations_params):
        alpha = params['alpha']
        # Discard burn_in steps to reach the stationary state
        stationary_data = data[:, burn_in:].flatten()
        sns.kdeplot(stationary_data, label=f'$\\alpha = {alpha}$', fill=True, alpha=0.2)

    plt.axvline(0, color='black', linestyle='--', alpha=0.5)
    plt.title("Stationary distribution of the AR(1) process")
    plt.xlabel("State value $X_n$")
    plt.ylabel("Density")
    plt.legend(title="Coefficient $\\alpha$")
    plt.grid(True, alpha=0.3)
    plt.show()

### Exercise 2.6 


#### Theoretical solutions 
##### Stationary distribution 
We consider following recurrence
$$\begin{align*}X_{n+1} & = \alpha X_n + W_{n+1} & W_{n+1} \sim \mathcal{N}(0, v^2)  \end{align*}$$
To find stationary distribution $f$ we need to solve:
$$f(y) = \int_\mathbb{R} f(x) \cdot p(y|x) \mathrm{d}x$$
From recurrence we obtain that $p(y|x) \sim \mathcal{N}(\alpha x, v^2)$. Because $X_0$ is normal and sum of normal distributions is normal distribution we assume that $f \sim \mathcal{N}({\mu, \sigma^2})$. We assume that $X_n \sim \mathcal{N}(\mu, \sigma^2)$ and want to derive parameters such that $X_{n+1}=X_n$. We can derive formula for $f$ in two ways:
1. Solving above equation
2. Use moments methods, because we know that normal distribution is unambiguous to mean and variance:
$$\begin{align*} \mathbb{E}[X_{n+1}] &= \mathbb{E}[\alpha X_{n} + W_{n+1}] = \alpha \mu+0=\mu \Leftrightarrow \mu(\alpha-1)=0 \\ 
&\implies \mu=0\\
\mathrm{Var}(X_{_{n+1}}) &= \mathrm{Var}(\alpha X_{n} + W_{n+1}) = \alpha^{2}\sigma^{2} + v^{2} =\sigma^{2} \Leftrightarrow \sigma^{2} \\
& \implies \sigma^{2} = \frac{v^{2}}{1-\alpha^{2}}\end{align*}$$
That is stationary distribution $f$ is $\mathcal{N}(0, \frac{v^{2}}{1-\alpha^{2}})$

##### Marginal distribution $X_n|X_0$
From recurrence we obtain that:
$$\begin{align}
X_{n+1}  & = \alpha X_{n} + W_{n+1} = \alpha(\alpha X_{n-1} + W_{n})= \\
 & =\alpha^{2}X_{n-1} + \alpha W_{n} + W_{n+1} =  \\
 & \vdots  \\
 & = \alpha^{n+1}X_{0} + \sum_{i=0}^{n+1} \alpha^{i}W_{n+1-i}   
\end{align}$$



In [ ]:
def ar_process(x_0, traj_length: int, probe_size: int, params: dict):
    """
    Simulates an AR(1) process: X_{n+1} = alpha * X_n + W_{n+1}
    x_0: Initial state(s)
    traj_length: Number of time steps
    probe_size: Number of independent simulations (trajectories)
    params: Dictionary containing 'alpha' (coefficient) and 'sigma' (noise std dev)
    """
    alpha = params['alpha']
    sigma = params['sigma']

    # Generate white noise W_0, ... W_n ~ N(0, sigma^2)
    W = stats.norm(0, sigma).rvs([probe_size, traj_length])
    
    # Use lfilter to solve the linear difference equation
    # lfilter([1], [1, -alpha], W) implements the AR(1) recursion
    X_sum = signal.lfilter([1], [1, -alpha], W, axis=1)

    # Calculate the influence of the initial value over time: x_0 * alpha^n
    powers = alpha ** np.arange(1, traj_length + 1)
    
    if np.isscalar(x_0):
        X_init = x_0 * powers
    else:
        # Reshape for broadcasting across multiple trajectories
        X_init = np.atleast_1d(x_0)[:, np.newaxis] * powers

    return X_init + X_sum

In [ ]:
# Parameters
traj_length = 1000
probe_size = 1
params = {'alpha': 0.9, 'sigma': 1}

# Simulation: Start from a random initial state N(10, 2^2)
x0_initial = stats.norm(10, 2).rvs(probe_size)
ar_probe = ar_process(x0_initial, traj_length=traj_length, 
                      probe_size=probe_size, params=params)

# Stationary standard deviation: sigma / sqrt(1 - alpha^2)
sigma_stat = (params['sigma']**2 / (1 - params['alpha']**2))**0.5
barriers = [-5 * sigma_stat, 5 * sigma_stat]

plot_trajectories_barriers(ar_probe, barriers, f'AR process simulation for {params}')
plt.show()

In [ ]:
# params
traj_length = 1000
probe_size = 1
params = {
    'alpha': -0.9,
    'sigma': 1, 
}

## probe
init_var_stat = params['sigma']**2 / (1 - params['alpha']**2) 
ar_probe = ar_process(stats.norm(0, init_var_stat).rvs(probe_size), traj_length=traj_length, probe_size=probe_size, params=params)

## plot 
sigma_2 = (params['sigma'] / (1 - params['alpha']**2))**(1/2)
barriers = [- 5*sigma_2, 5 * sigma_2]
plot_trajectories_barriers(ar_probe, barriers, f'Ar process barriers for {params}')
plt.show()

##### Law of large numbers for Markov chains

The states that for long enough trajectory we obtain:
$$\frac{1}{n} \sum_{i=1}^n \mathbb{1}(X_i\in A) \xrightarrow{\text{a.s}} \pi(A)$$

In [ ]:
## simulation of stationary process

traj_length = 300_000
probe_size = 1
simulations_params = [
    {'alpha': i/10, 'sigma': 1} for i in range(1, 10)
]

probes = [
    ar_process(stats.norm(3, 1).rvs(probe_size), traj_length=traj_length, probe_size=probe_size, params=params) for params in simulations_params
]


plot_ar_comparison(probes, simulations_params, burn_in=100)
